# QLoRA trên Colab — ĐATN Căn Cứ

Notebook này chỉ là lớp vỏ gọi `qlora_train.py`; toàn bộ logic nằm trong script để
chạy ở Colab hay máy rời đều cho **cùng một kết quả**.

**Trước khi chạy:** Runtime → Change runtime type → GPU (T4 16GB đủ cho 7–8B QLoRA).

Cần mang sang 3 file: `qlora_train.py`, `train.jsonl`, `validation.jsonl`.
Quy trình đầy đủ và cách bàn giao ngược: xem `training/README.md`.

In [ ]:
!nvidia-smi
!pip -q install transformers==4.48.0 peft==0.14.0 bitsandbytes==0.45.0 accelerate==1.2.1

## 1. Nạp dữ liệu và script

Chạy ô dưới rồi chọn `qlora_train.py`, `train.jsonl`, `validation.jsonl` từ máy.
Dữ liệu lớn thì mount Google Drive thay cho upload tay.

In [ ]:
from google.colab import files

files.upload()

# Cách khác — dữ liệu để sẵn trên Drive:
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/datn/{qlora_train.py,train.jsonl,validation.jsonl} .

## 2. Smoke test (~2 phút)

Kiểm tra torch + bitsandbytes + peft khớp nhau **trước khi** tốn giờ GPU cho lượt thật.

In [ ]:
!python qlora_train.py --smoke --base-model Qwen/Qwen2.5-1.5B-Instruct --out /content/smoke

## 3. Huấn luyện thật

Không gian tìm kiếm theo Plan/03 §3: `rank ∈ {8, 16, 32}`, `lr` 1e-5 → 2e-4, `epochs` 2–4.
Mỗi lượt đổi `--out` sang tên khác để so sánh được cả loạt.
**Chọn cấu hình theo eval loss trên validation, không theo train loss.**

In [ ]:
!python qlora_train.py \
    --train train.jsonl --val validation.jsonl \
    --base-model Qwen/Qwen2.5-7B-Instruct \
    --rank 16 --alpha 32 --lr 2e-4 --epochs 3 \
    --max-seq-len 1536 \
    --out /content/qwen25-7b-r16

## 4. Tải adapter về

Giải nén vào `backend/models/adapters/` là cấu hình C/D chạy được — không sửa code,
không chạy migration. Nhớ mở `adapter_card.json` xem `eval_loss` trước khi báo kết quả.

In [ ]:
import json

print(json.dumps(json.load(open('/content/qwen25-7b-r16/adapter_card.json')), ensure_ascii=False, indent=2))

!rm -rf /content/qwen25-7b-r16/_checkpoints
!zip -qr /content/qwen25-7b-r16.zip /content/qwen25-7b-r16
files.download('/content/qwen25-7b-r16.zip')